# Backpropagation and Training Neural Networks, Explained Simply
The chain rule by hand, learning rates, mini-batches, overfitting, early stopping and dropout, in PyTorch. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
1,797 handwritten digits that come with scikit-learn. Each is a tiny 8 x 8 grey image: 64 numbers from 0 (white) to 16 (black).

### 1,797 handwritten digits
- `load_digits`: a small dataset built into scikit-learn, no download needed
- `X.shape`: 1797 images, 64 pixel values each
- `y`: the digit each image shows, 0 to 9

In [ ]:
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)
print(X.shape)
print("first labels:", y[:10])

### Look at some digits
- `img.reshape(8, 8)`: turn 64 numbers back into an 8 x 8 picture
- `cmap="gray_r"`: 0 is white, 16 is black

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, img, label in zip(axes, X, y):
    ax.imshow(img.reshape(8, 8), cmap="gray_r")
    ax.set_title(label); ax.axis("off")
plt.show()

## 2. Backpropagation by hand
Prediction = w x x, loss = (prediction - target)^2. The slope of the loss for w is **2 x (prediction - target) x x**. Then **new w = w - learning rate x slope**. With two weights, the **chain rule** multiplies the slopes along the path.

### One weight, one step
- `requires_grad=True`: ask PyTorch to track this number so it can find its slope
- `loss.backward()`: backpropagation: compute the slope
- `w.grad`: 2 x (6 - 10) x 2 = -16
- `w -= 0.05 * w.grad`: one gradient descent step: 3 + 0.8 = 3.8

In [ ]:
import torch

x, target = 2.0, 10.0
w = torch.tensor(3.0, requires_grad=True)
pred = w * x
loss = (pred - target) ** 2
loss.backward()
print("prediction:", pred.item(), " loss:", loss.item())
print("slope dL/dw:", w.grad.item())
with torch.no_grad():
    w -= 0.05 * w.grad
print("new w:", round(w.item(), 2))

### Two weights: the chain rule
- `h = w1 * x`: a hidden value: 1.5 x 2 = 3
- `pred = w2 * h`: 2 x 3 = 6
- `w2.grad`: -8 x h = -8 x 3 = -24
- `w1.grad`: -8 x w2 x x = -8 x 2 x 2 = -32

In [ ]:
w1 = torch.tensor(1.5, requires_grad=True)
w2 = torch.tensor(2.0, requires_grad=True)
h = w1 * x
pred = w2 * h
loss = (pred - target) ** 2
loss.backward()
print("h:", h.item(), " prediction:", pred.item())
print("slope for w2:", w2.grad.item())
print("slope for w1:", w1.grad.item())

## 3. From scores to probabilities
With 10 classes the network gives 10 scores. **Softmax** turns them into probabilities that add up to 1. The **cross-entropy loss** is -log(probability of the right answer).

### Softmax and cross-entropy
- `torch.softmax`: e^score for each, divided by their total
- `-torch.log(probs[0])`: small when the right class gets a high probability

In [ ]:
scores = torch.tensor([2.0, 1.0, 0.1])
probs = torch.softmax(scores, dim=0)
print("probabilities:", probs.round(decimals=2))
print("loss if the answer is class 0:",
      round(-torch.log(probs[0]).item(), 3))

## 4. Prepare the data
Scale the pixels to 0 to 1, then split: 60% to train, 20% to check during training (validation), 20% for the final test.

### Scale, split, convert
- `X / 16`: pixels from 0 to 1
- `stratify=y`: every digit in the same proportion in each part
- `test_size=0.25`: a quarter of the remaining 80% = 20% of all, for validation

In [ ]:
from sklearn.model_selection import train_test_split

X = X / 16
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr, y_tr, test_size=0.25, random_state=0, stratify=y_tr)
X_tr, X_va, X_te = [torch.tensor(a, dtype=torch.float32)
                    for a in (X_tr, X_va, X_te)]
y_tr, y_va, y_te = [torch.tensor(a) for a in (y_tr, y_va, y_te)]
print(len(X_tr), len(X_va), len(X_te))

## 5. The training loop
A network with two hidden layers of 256 neurons. Training goes in **mini-batches** of 32 images; one pass over all training images is an **epoch**.

### The network
- `nn.Linear(64, 256)`: 256 neurons, each looking at all 64 pixels
- `nn.Dropout(dropout)`: switched off for now (0.0); used later
- `nn.Linear(256, 10)`: 10 outputs: one score per digit

In [ ]:
import copy
import torch.nn as nn

def make_net(dropout=0.0):
    torch.manual_seed(0)
    return nn.Sequential(
        nn.Linear(64, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 10))

print("weights and biases:",
      sum(p.numel() for p in make_net().parameters()))

### The training function
- `CrossEntropyLoss`: softmax plus cross-entropy in one
- `torch.randperm`: shuffle the training images every epoch
- `range(0, len(X_tr), 32)`: mini-batches of 32 images
- `zero_grad / backward / step`: clear old slopes, backpropagate, nudge the weights
- `model.train() / model.eval()`: dropout on while training, off while checking
- `best = copy.deepcopy(...)`: early stopping: remember the weights from the best validation epoch, and restore them at the end

In [ ]:
def train(model, optimizer, epochs):
    loss_fn = nn.CrossEntropyLoss()
    train_hist, val_hist = [], []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), 32):
            idx = order[i:i + 32]
            optimizer.zero_grad()
            loss = loss_fn(model(X_tr[idx]), y_tr[idx])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_hist.append(loss_fn(model(X_tr), y_tr).item())
            val_hist.append(loss_fn(model(X_va), y_va).item())
        if val_hist[-1] == min(val_hist):
            best = copy.deepcopy(model.state_dict())
    model.load_state_dict(best)
    return train_hist, val_hist

## 6. The learning rate
The learning rate is the size of each step. Too small: painfully slow. Too big: it overshoots and never settles. **Adam** adjusts the step for each weight automatically.

### Compare learning rates
- `getattr(torch.optim, opt)`: pick torch.optim.SGD or torch.optim.Adam by name
- `SGD`: plain gradient descent on mini-batches
- `the picture`: 0.001 barely moves, 3 never learns, 0.1 and Adam learn fast

In [ ]:
runs = [("SGD, lr 0.001", "SGD", 0.001), ("SGD, lr 0.1", "SGD", 0.1),
        ("SGD, lr 3", "SGD", 3.0), ("Adam, lr 0.001", "Adam", 0.001)]
for name, opt, lr in runs:
    model = make_net()
    optimizer = getattr(torch.optim, opt)(model.parameters(), lr=lr)
    _, val = train(model, optimizer, epochs=20)
    print(f"{name:<15} validation loss after 20 epochs: {val[-1]:.3f}")
    plt.plot(range(1, 21), val, label=name)
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.legend()
plt.show()

## 7. Overfitting and early stopping
Train for 100 epochs and watch both losses. When the validation loss starts rising while the training loss keeps falling, the network is memorising: **overfitting**.

### Train for 100 epochs
- `np.argmin(va)`: the epoch with the lowest validation loss
- `plt.ylim(0, 0.6)`: zoom in on the interesting part
- `plt.axvline`: a dashed line at the best epoch: early stopping keeps these weights
- `the picture`: training loss falls to almost 0; validation loss turns back up

In [ ]:
import numpy as np

plain = make_net()
tr, va = train(plain, torch.optim.Adam(plain.parameters(), lr=0.001), 100)
best = int(np.argmin(va))
print(f"best epoch: {best + 1}, validation loss {va[best]:.3f}")
print(f"epoch 100: validation loss {va[-1]:.3f}, training loss {tr[-1]:.4f}")
plt.plot(range(1, 101), tr, label="training")
plt.plot(range(1, 101), va, label="validation")
plt.axvline(best + 1, color="gray", linestyle="--")
plt.ylim(0, 0.6); plt.xlabel("epoch"); plt.ylabel("loss")
plt.legend(); plt.show()

## 8. Dropout
**Dropout** randomly switches off half the neurons at every training step, so the network can't rely on any single one. It's switched off when predicting.

### Train with dropout
- `make_net(dropout=0.5)`: the same network, but half the hidden neurons dropped at each step
- `the picture`: the dropout curve keeps improving instead of turning back up

In [ ]:
drop = make_net(dropout=0.5)
tr_d, va_d = train(drop, torch.optim.Adam(drop.parameters(), lr=0.001), 100)
print(f"with dropout, epoch 100: validation loss {va_d[-1]:.3f}")
plt.plot(range(1, 101), va, label="no dropout")
plt.plot(range(1, 101), va_d, label="dropout 0.5")
plt.ylim(0, 0.6); plt.xlabel("epoch"); plt.ylabel("validation loss")
plt.legend(); plt.show()

### Final test
- `argmax(1)`: the digit with the highest score
- `X_te`: 360 images the network has never seen

In [ ]:
with torch.no_grad():
    for name, m in [("early stopping", plain), ("dropout", drop)]:
        acc = (m(X_te).argmax(1) == y_te).float().mean().item()
        print(f"{name:<15} test accuracy: {acc:.3f}")

## 9. Mini project: read some digits
Show test images with the network's guess.

### Read ten test digits
- `guess`: the network's answer for the first 10 test images
- `color=...`: green if right, red if wrong

In [ ]:
with torch.no_grad():
    guess = drop(X_te[:10]).argmax(1)
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for ax, img, g, t in zip(axes, X_te[:10], guess, y_te[:10]):
    ax.imshow(img.reshape(8, 8), cmap="gray_r")
    ax.set_title(f"{g.item()}", color="green" if g == t else "red")
    ax.axis("off")
plt.show()

## Try it yourself
1. Work out the one-weight example by hand for w = 4. Then check it with the code.
2. Try a learning rate of 0.5 with SGD. Is it closer to 0.1 or to 3?
3. Change the dropout from 0.5 to 0.2. What happens to the validation loss?